In [ ]:
from scdesigner.datasets import pancreas

example_sce = pancreas()
example_sce

In [ ]:
from scdesigner.simulators import NegBinCopula
from scdesigner.transform import nullify, amplify, replace_param

sim = NegBinCopula("~ bs(pseudotime, degree=6)")
sim.fit(example_sce)
outcomes = example_sce.var_names[:4]
mask = replace_param(sim.params["coef_mean"], ["pseudotime"], outcomes)

In [ ]:
from copy import deepcopy

null_sim = deepcopy(sim)
null_sim.params = nullify(sim.params, "coef_mean", mask)
null_sim.params["coef_mean"]

In [ ]:
samples = null_sim.sample(example_sce.obs)
samples.X[:10, :10]

In [ ]:
from scdesigner.diagnose import compare_means, compare_standard_deviation, compare_umap
import numpy as np

log_relative = lambda x: np.log1p(x / x.sum(axis=1, keepdims=True))
compare_means(example_sce, samples, log_relative)


In [ ]:
compare_standard_deviation(example_sce, samples, log_relative)

In [ ]:
compare_umap(example_sce, samples, transform=log_relative)

In [ ]:
sim = NegBinCopula(epochs=10)
sim.fit(example_sce, "~ bs(pseudotime, degree=2)")
mask = data_frame_mask(sim.params["covariance"], outcomes)

null_sim = deepcopy(sim)
null_sim.params = nullify(sim.params, "covariance", mask)
null_sim.params["covariance"]

In [ ]:
mask = data_frame_mask(sim.params["coef_mean"], ["pseudotime"], outcomes)
null_sim.params = amplify(sim.params, "coef_mean", mask, factor=2)
null_sim.params["coef_mean"] / sim.params["coef_mean"]

In [ ]:
sim.sample(example_sce.obs)